# Penalty sweep over subsamples -- train & plot directly

The pure path: no cache, no SLURM. Pick a target + method, sweep penalties over the full data
**plus k bootstrap subsamples**, and plot one factor's beta-vs-penalty. The cache-backed path
(`sb.compute_sweep_dataset` on SLURM + `sb.plot_cached_sweep_factor`) is shown at the bottom.

## Load + set up factors

In [ ]:
import sys
from pathlib import Path
_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import scanpy as sc
from metab_processing.LinearRegression import signature_helpers as sh, signature_batch as sb
from metab_processing.LinearRegression.signature_configs import CONFIGS
from metab_processing.LinearRegression.build_x import ensure_lognorm_layer

cfg = CONFIGS['breast']                       # dataset
adata = sc.read_h5ad(cfg.adata_path); ensure_lognorm_layer(adata)
var = set(adata.var_names)

library = sh.load_msigdb(cfg.dbver, cfg.categories)
# cancer gene sets as neighborhood factors (so they're available as predictors)
cancer_sets = {n: [g for g in library.get(n, []) if g in var] for n in cfg.cancer_factors}
cancer_sets.update({v: [g for g in gl if g in var] for v, gl in cfg.vessel.items()})
sh.add_cancer_factors(adata, {k: v for k, v in cancer_sets.items() if v}, annot_col=cfg.cell_type_col)

# pooled target cells (T cells for cancers / IEC for colon), no covariate
cts = cfg.sweep_cell_types or (list(cfg.t_cell_types) if cfg.kind == 'cancer' else [cfg.cell_type])
sh.make_group_column(adata, cfg.cell_type_col, cts, group_name='_sweep_group', group_label='GROUP')
print('pooled cell types:', cts)

## Train: sweep penalties over full + k subsamples

In [ ]:
SIG = 'HALLMARK_HYPOXIA'          # any target signature
METHOD = 'l1'                     # 'l1' | 'ridge' | 'elastic'  (penalties are per-method)
PENALTIES = cfg.sweep_penalties[METHOD]

genes = [g for g in library.get(SIG, []) if g in var]
ad2 = sh.add_gene_signature(adata, SIG, positive=genes, keep_when_overlap='factor', verbose=False)
sweep = sh.sweep_subsample_betas(ad2, SIG, PENALTIES, n_subsamples=cfg.sweep_k,
                                 annot_col='_sweep_group', annot_value='GROUP',
                                 metabolites=[], custom='all', method=METHOD,
                                 l1_ratio=cfg.l1_ratio, standardize=cfg.standardize)
print('subsamples:', sorted(sweep['subsample'].unique(), key=str), '| penalties:', PENALTIES)
sweep.head()

## Plot one factor (full = bold, subsamples = faint)

In [ ]:
# pick a factor to inspect (edit); e.g. the strongest at the smallest penalty:
import numpy as np
cand = (sweep[sweep['subsample'] == 'full']
        .assign(absb=lambda d: d['beta'].abs()).sort_values('absb', ascending=False))
print('some factors:', list(cand['factor'].drop_duplicates().head(10)))
FACTOR = cand['factor'].iloc[0]
sh.plot_sweep_factor(sweep, FACTOR);

## Cache-backed path (SLURM)

Compute once on SLURM (`signature_dispatch.ipynb` -> penalty-sweep cells, or
`python signature_sweep_run.py --config breast --method l1`), then plot from the cache:
```python
sb.plot_cached_sweep_factor(cfg, 'l1', 'HALLMARK_HYPOXIA', FACTOR)
```

In [ ]:
# sb.plot_cached_sweep_factor(cfg, METHOD, SIG, FACTOR)   # uncomment once the sweep is cached